# Last-block attention at 336 px: inference latency

Load the existing `last_blocks_20261010_022115/last.pt` checkpoint and measure single-view forward-pass latency. **No training, dataset download or prediction submission.**

In Colab, select **Runtime → Change runtime type → A100 GPU**, then run the cells in order. This matches the GPU used in the presentation comparison. Other GPUs produce valid measurements for that hardware, but are not directly comparable.

Protocol: BF16, batch sizes 1 and 64, 30 warm-up iterations and 200 timed iterations. Timing excludes checkpoint loading, preprocessing, data transfer, flip TTA and zoom. Results save to the existing run folder on Drive.

## 1. Settings
The saved run configuration determines the model architecture and resolution. The notebook checks these settings rather than changing a 224 px model to 336 px.

In [ ]:
from pathlib import Path

RUN_DIR = Path(
    "/content/drive/MyDrive/CS604_RUNS/last_blocks_20261010_022115"
)
CKPT = RUN_DIR / "last.pt"

REPO_URL = "https://github.com/keaidewoq/cs604-project.git"
BRANCH = "main"
REPO_DIR = Path("/content/cs604_latency_repo")
WARMUP = 30
ITERATIONS = 200
BATCH_SIZES = (1, 64)

## 2. Mount Drive and check the run folder
Mount the Drive account containing this folder, or add the shared CS604_RUNS folder as a shortcut to My Drive.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
assert RUN_DIR.is_dir(), f"Run folder not found: {RUN_DIR}. Check the Drive account or shared-folder shortcut."
assert CKPT.is_file(), f"Checkpoint not found: {CKPT}"
assert (RUN_DIR / "config.yaml").is_file(), f"Saved configuration not found: {RUN_DIR / 'config.yaml'}"
print("Checkpoint:", CKPT)

## 3. Get the repository and install dependencies
Use a GitHub token if the repository needs authentication. Leave the hidden prompt empty for public access. The repository copy is separate from any other Colab project checkout.

In [ ]:
import base64
import os
import subprocess
import sys
from getpass import getpass

token = getpass("GitHub token (hidden; leave empty for public access): ")
env = os.environ.copy()
if token:
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    env.update(
        GIT_CONFIG_COUNT="1",
        GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
        GIT_CONFIG_VALUE_0=f"Authorization: Basic {auth}",
    )
try:
    if (REPO_DIR / ".git").is_dir():
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], env=env, check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "merge", "--ff-only", f"origin/{BRANCH}"], check=True)
    else:
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], env=env, check=True)
finally:
    del token, env
    if "auth" in globals():
        del auth

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
REPO_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Repository commit:", REPO_COMMIT)

## 4. Load the trained 336 px model
A checkpoint containing only trainable parameters also needs the original pretrained backbone, which timm downloads automatically if it is not cached. The classifier weights come from the checkpoint, so a separate prototype file is not needed for this measurement.

In [ ]:
import gc
import json
import torch

from src.utils import load_config, apply_overrides, set_seed
from src.model import build_model
from scripts.benchmark_latency import measure

assert torch.cuda.is_available(), "Select a GPU in Runtime → Change runtime type."
assert torch.cuda.is_bf16_supported(), "Select a GPU supporting BF16, preferably A100."

cfg = load_config(RUN_DIR / "config.yaml")
assert int(cfg.img_size) == 336, f"This run is {cfg.img_size} px, not the requested 336 px."
assert cfg.method == "last_blocks", f"Unexpected method: {cfg.method}"
assert int(cfg.get("last_n_blocks", 2)) == 1, "Expected last-block attention, not multiple blocks."
assert cfg.get("train_scope", "full") == "attention", "Expected full attention scope: QKV plus output projection."
assert bool(cfg.bf16), "The saved run does not enable BF16; inspect the configuration before comparing timings."

# Release a model from any previous execution before rebuilding.
if "model" in globals():
    del model
gc.collect()
torch.cuda.empty_cache()

checkpoint = torch.load(CKPT, map_location="cpu", weights_only=False)
partial = bool(checkpoint.get("trainable_only", False)) if isinstance(checkpoint, dict) else False
state = checkpoint["state_dict"] if isinstance(checkpoint, dict) and "state_dict" in checkpoint else checkpoint
apply_overrides(cfg, pretrained=partial)
set_seed(int(cfg.get("seed", 42)))

device = torch.device("cuda")
model = build_model(cfg).to(device)
expected = set(model.state_dict())
provided = set(state)
assert not provided - expected, f"Unexpected checkpoint keys: {sorted(provided - expected)[:8]}"
trainable_keys = {name for name, param in model.named_parameters() if param.requires_grad}
assert trainable_keys <= provided, f"Trained parameters missing from checkpoint: {sorted(trainable_keys - provided)[:8]}"
incompatible = model.load_state_dict(state, strict=not partial)
model.eval()
del checkpoint, state
gc.collect()
torch.cuda.empty_cache()

GPU_NAME = torch.cuda.get_device_name(0)
print("GPU:", GPU_NAME)
print("Resolution:", cfg.img_size)
print("Method:", cfg.method, "| scope:", cfg.train_scope, "| last blocks:", cfg.last_n_blocks)
print("Head:", cfg.get("head_type", "linear"))
print("Checkpoint contains only trainable parameters:", partial)
print("Frozen parameters restored from pretrained backbone:", len(incompatible.missing_keys))
if "A100" not in GPU_NAME:
    print("Hardware differs from the slide comparison: record this GPU name with the result.")

## 5. Benchmark and save
Run this cell again to repeat the timing without rebuilding the model. The batch-1 `ms_per_image` value fills the missing slide-9 latency. At batch 64, `ms_per_image` is throughput expressed per image, while `ms_per_batch` is the duration of the full batch.

The latest result is `inference_latency.json`. A timestamped copy preserves each execution. Compare small latency differences cautiously because timings vary between sessions.

In [ ]:
from datetime import datetime, timezone

assert WARMUP > 0 and ITERATIONS > 0
assert tuple(BATCH_SIZES) == (1, 64), "Keep batches 1 and 64 to match the comparison protocol."
assert int(cfg.img_size) == 336
results = {
    "run_dir": str(RUN_DIR),
    "checkpoint": str(CKPT),
    "measured_at_utc": datetime.now(timezone.utc).isoformat(),
    "repo_commit": REPO_COMMIT,
    "method": cfg.method,
    "train_scope": cfg.train_scope,
    "last_n_blocks": int(cfg.last_n_blocks),
    "head_type": cfg.get("head_type", "linear"),
    "gpu": GPU_NAME,
    "gpu_capacity_GiB": torch.cuda.get_device_properties(0).total_memory / 1024**3,
    "torch_version": str(torch.__version__),
    "cuda_version": torch.version.cuda,
    "precision": "bf16",
    "img_size": int(cfg.img_size),
    "warmup_iterations": WARMUP,
    "timed_iterations": ITERATIONS,
    "scope": "Single-view forward pass on synthetic inputs; excludes checkpoint loading, preprocessing, transfers, flip TTA and zoom",
    "measurements": [],
}

for batch_size in BATCH_SIZES:
    batch = torch.randn(batch_size, 3, 336, 336, device=device)
    try:
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            ms_per_image, _ = measure(model, batch, device, WARMUP, ITERATIONS)
        results["measurements"].append({
            "batch_size": batch_size,
            "ms_per_image": ms_per_image,
            "ms_per_batch": ms_per_image * batch_size,
        })
        print(f"Batch {batch_size}: {ms_per_image:.4f} ms/image, {ms_per_image * batch_size:.3f} ms/batch")
    finally:
        del batch

timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
output = RUN_DIR / "inference_latency.json"
archive = RUN_DIR / f"inference_latency_336_{timestamp}.json"
payload = json.dumps(results, indent=2)
archive.write_text(payload, encoding="utf-8")
output.write_text(payload, encoding="utf-8")

bs1, bs64 = results["measurements"]
print(f"\nSlide 9, last-block attention at 336 px: {bs1['ms_per_image']:.2f} ms (batch 1)")
print("Saved:", output)
print("Archived:", archive)
print("\n" + payload)